# 03 · 自定义工具（Custom Tools）

目标：用 `@define_tool` + Pydantic 注册自定义工具；理解 built-in tool 覆写与权限跳过。

**心智模型**：当你把工具传给 `create_session(tools=[...])`，CLI 会在每轮 LLM 决定调用工具时发送 `tool.call` RPC，SDK 自动执行你的 handler（sync 或 async），把返回值序列化回去。

## 0. 工具体系全景图

SDK 里能被 LLM 调用的「工具」有 **4 个来源**，全都汇入 `create_session(tools=..., mcp_servers=...)` 这一道入口：

```mermaid
flowchart LR
    subgraph App["你提供的"]
        Decorated["@define_tool<br/>(Pydantic 自动 schema)"]
        Raw["Tool(name=...,<br/>parameters={...},<br/>handler=fn)"]
        DeclOnly["Tool(handler=None)<br/>(declaration-only)"]
    end
    subgraph Builtin["CLI 自带"]
        B15["15 个内建工具<br/>bash · view · apply_patch · rg · glob · web_fetch ..."]
    end
    subgraph MCP["MCP 协议"]
        M1["stdio MCP server<br/>(@modelcontextprotocol/server-filesystem ...)"]
        M2["http/sse MCP server<br/>(learn.microsoft.com/api/mcp ...)"]
    end

    Decorated --> Session["session.create_session(<br/>tools=[...],<br/>mcp_servers={...},<br/>available_tools=[...],<br/>excluded_tools=[...])"]
    Raw --> Session
    DeclOnly --> Session
    B15 --> Session
    M1 --> Session
    M2 --> Session
    Session --> LLM["LLM 看到的<br/>统一工具列表"]
```

| 来源 | 谁执行 | 适合 |
|---|---|---|
| `@define_tool` + Pydantic | SDK 在你的进程内同步调 Python 函数 | 99% 业务自定义工具的首选 |
| `Tool(handler=fn)` 低阶 API | 同上 | 已有 OpenAPI / JSON Schema，想绕开 Pydantic |
| `Tool(handler=None)` declaration-only | **不执行**，外部代码收 `external_tool.requested` 事件后回填 | 慢工具 / 跨进程 / 人在回路 |
| 内建工具 | CLI 子进程内执行 | 文件、shell、grep、HTTP fetch（详见 notebook 07）|
| `mcp_servers={...}` | CLI 通过 MCP 协议路由 | 复用社区 MCP，零代码接入 |

## 0.1 一次自定义工具调用的时序

```mermaid
sequenceDiagram
    autonumber
    participant LLM
    participant CLI as copilot CLI
    participant SDK as Python SDK
    participant App as on_permission_request
    participant Tool as @define_tool 函数

    LLM->>CLI: tool_call(name=lookup_issue, args={...})
    CLI->>SDK: rpc: tool.call
    SDK->>App: PermissionRequest(kind=custom-tool)
    App-->>SDK: PermissionRequestResult(approve-once)
    Note over SDK,Tool: skip_permission=True 时跳过权限步骤
    SDK->>Tool: 调 handler(params: Pydantic)
    Tool-->>SDK: 返回 str / ToolResult
    SDK-->>CLI: ToolCallResult
    CLI-->>App: event: tool.executionComplete<br/>(序列化后字段叫 .content)
    CLI->>LLM: 把 tool 结果回灌
    LLM-->>CLI: 最终回复
```

**3 个高频踩坑**（更多见 [README.md](README.md) §5）：

- `ToolResult(text_result_for_llm=...)` 是**入参**字段名；订阅事件时它叫 `event.data.result.content`
- 想覆写同名内建工具，必须显式 `@define_tool(name='view', overrides_built_in_tool=True)`
- `available_tools` / `excluded_tools` **只对内建工具生效**，不影响你 `tools=[...]` 注册的自定义工具


## 1. 推荐写法：`@define_tool` + Pydantic

- 自动从 Pydantic model 推导 JSON Schema
- 注意：若使用 `from __future__ import annotations`，必须把 Pydantic 模型放在 **模块顶层**（不能在函数内定义）

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from copilot import CopilotClient, define_tool
from copilot.session import PermissionHandler
from copilot.generated.session_events import (
    AssistantMessageData, SessionIdleData, 
    ToolExecutionStartData, ToolExecutionCompleteData
)

# 加载 .env 获取 GPT 5.4 端点
load_dotenv()

azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}

class LookupIssueParams(BaseModel):
    id: str = Field(description='Issue identifier, e.g. AGILE-123')

@define_tool(description='Fetch issue details from our tracker')
async def lookup_issue(params: LookupIssueParams) -> str:
    # 模拟一个实际的代码运行
    print(f'\n[🔥 1. 自定义工具触发] lookup_issue 正在查询系统，目标 ID: {params.id}')
    return f'Issue {params.id} — status=open, owner=alice, title="Fix login bug"'

async def demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[lookup_issue],
        ) as session:
            # 💡 【多监听模式选择】：
            # 写法一（最简一键式）：直接使用 `session.send_and_wait(...)` 发送并等待空闲，安全获取最终回复文本。
            response = await session.send_and_wait('Use lookup_issue to summarize AGILE-123 in one sentence.')
            if response and hasattr(response.data, 'content'):
                print('\n[💬 最终回复]:', response.data.content)

            # 写法二（事件监听模式）：如果您未来需要捕获细粒度的流式数据、工具中间态以及原始 RPC 报文：
            # done = asyncio.Event()
            
            # def on_event(event):
            #     # 打印会话过程中底层的各类原始 RPC 或高级 Event 报文
            #     match event.data:
            #         case ToolExecutionStartData() as d:
            #             args = getattr(d, 'tool_args', getattr(d, 'arguments', 'N/A'))
            #             print(f'\n[🔧 2. ToolExecutionStart] 大模型决定调用自定义工具 lookup_issue，入参: {args}')
            #         case ToolExecutionCompleteData() as d:
            #             res = getattr(d.result, 'content', 'N/A') if d.result else 'N/A'
            #             print(f'[✅ 3. ToolExecutionComplete] 自定义工具执行完毕，输出: {res}')
            #         case AssistantMessageData() as d:
            #             print('\n[💬 最终回复]:', d.content)
            #         case SessionIdleData():
            #             # 当流式应答输出完毕，整个会话到达 Idle 闲置状态，触发信号解锁 done.wait() 以退出执行
            #             done.set()
            
            # # 注册会话级监听器
            # session.on(on_event)
            
            # # 触发会话并等待闲置信号
            # await session.send('Use lookup_issue to summarize AGILE-123 in one sentence.')
            # await done.wait()

await demo()

## 2. 低阶 API：`Tool` + 手写 schema

适合从已有 OpenAPI / 历史工具迁移。

In [ ]:
from copilot.tools import Tool, ToolInvocation, ToolResult

async def _lookup(invocation: ToolInvocation) -> ToolResult:
    issue_id = invocation.arguments['id']
    return ToolResult(
        text_result_for_llm=f'Issue {issue_id} resolved.',
        result_type='success',
        session_log=f'fetched {issue_id}',
    )

raw_tool = Tool(
    name='lookup_issue_raw',
    description='Fetch issue (raw API)',
    parameters={
        'type': 'object',
        'properties': {'id': {'type': 'string'}},
        'required': ['id'],
    },
    handler=_lookup,
)

## 2.1 运行低阶 API 工具实验

验证由 `Tool` 手写的 schema 与 `ToolInvocation` 处理器的低门槛机制。

In [ ]:
async def run_raw_tool_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[raw_tool],
        ) as session:
            done = asyncio.Event()
            def on_event(event):
                match event.data:
                    case AssistantMessageData() as d:
                        print('Raw Tool Demo ASSISTANT:', d.content)
                    case SessionIdleData():
                        done.set()
            session.on(on_event)
            await session.send('Use lookup_issue_raw tool to resolve AGILE-456.')
            await done.wait()

await run_raw_tool_demo()

In [ ]:
async def run_raw_tool_demo_simple():
    """💡 极简一键式：用 send_and_wait 直接获取最终回复文本，无需手写事件循环。"""
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[raw_tool],
        ) as session:
            response = await session.send_and_wait('Use lookup_issue_raw tool to resolve AGILE-456.')
            if response and hasattr(response.data, 'content'):
                print('[💬 Raw Tool Demo - 最终回复]:', response.data.content)

await run_raw_tool_demo_simple()

## 2.2 对接极简 MCP 协议标准（Model Context Protocol）实验

根据 [Model Context Protocol (MCP)](https://modelcontextprotocol.io) 与微软规范，MCP 服务作为客户端或工具可以通过 JSON-RPC / IPC 交互。
以下提供了一个模拟的**嵌入式本地 MCP 转发桥**实验：
大模型调用 `@define_tool` 时，SDK 将工具入参转发至模拟的 MCP JSON-RPC 消息管道中（通过 `mcp:call` JSON-RPC），并返回最终结果。

In [ ]:
from pydantic import BaseModel, Field

# 1. 模拟一个标准的本地 MCP 服务管道运行逻辑
class MockMCPService:
    async def handle_json_rpc(self, request_payload: dict) -> dict:
        """模拟响应：JSON-RPC 2.0 -> mcp:call (https://learn.microsoft.com/api/mcp)"""
        if request_payload.get("method") == "tools/call":
            params = request_payload.get("params", {})
            name = params.get("name")
            arguments = params.get("arguments", {})
            
            if name == "get_mcp_doc":
                topic = arguments.get("topic", "general")
                # 模拟从微软或官方 MCP 提供的 API 文档
                return {
                    "jsonrpc": "2.0",
                    "id": request_payload.get("id"),
                    "result": {
                        "content": [
                            {
                                "type": "text",
                                "text": f"📚 [MCP Server Response] Topic: {topic}. Model Context Protocol (MCP) is an open standard that enables developers to build secure, bi-directional integrations between data sources, tools and AI models."
                            }
                        ]
                    }
                }
        return {"jsonrpc": "2.0", "error": {"code": -32601, "message": "Method not found"}}

# 初始化极简本地 MCP 端点
mcp_server = MockMCPService()

# 2. 定义契合 MCP Pydantic 的入参结构 
class MCPDocParams(BaseModel):
    topic: str = Field(description='The specific topic or URL of MCP docs, e.g. "architecture" or "https://learn.microsoft.com/api/mcp"')

# 3. 将本地工具封装为通往 MCP 服务的桥梁
@define_tool(description='Retrieve details from localized Model Context Protocol (MCP) documents endpoint.')
async def get_mcp_doc(params: MCPDocParams) -> str:
    # 模拟构建出标准的 JSON-RPC / MCP Request 
    mcp_request = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "get_mcp_doc",
            "arguments": {"topic": params.topic}
        }
    }
    
    # 通过管道转发给 MCP Service 核心
    print(f"\n[📡 Forwarding to MCP Server] Sending tools/call for query: {params.topic}")
    mcp_response = await mcp_server.handle_json_rpc(mcp_request)
    
    # 提取 MCP 规范返回的内容
    try:
        content_list = mcp_response["result"]["content"]
        texts = [item["text"] for item in content_list if item["type"] == "text"]
        return "\n".join(texts)
    except KeyError:
        return "Failed to parse standard MCP response."

# 4. 在 Session 级开始大模型与 MCP 工具的端到端调用测试
async def run_mcp_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[get_mcp_doc],
        ) as session:
            done = asyncio.Event()
            
            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        args = getattr(d, 'tool_args', getattr(d, 'arguments', 'N/A'))
                        print(f'[🔧 ToolExecutionStart] LLM chosen mcp tool. args: {args}')
                    case AssistantMessageData() as d:
                        print('\n[💬 ASSISTANT]:', d.content)
                    case SessionIdleData():
                        done.set()
                        
            session.on(on_event)
            # 让大模型调用我们的 MCP 自定义桥梁工具去查阅 microsoft-mcp API 信息
            await session.send('Use get_mcp_doc tool to query "https://learn.microsoft.com/api/mcp" details.')
            await done.wait()

await run_mcp_demo()

## 2.3 SDK 原生 MCP 一键接入 ⚡（推荐 / 正式生产用法）

其实 SDK 已内建 **MCP 协议支持**！直接通过 `create_session(mcp_servers={...})` 传入 MCP server 配置即可，CLI 会自动完成进程拉起、`tools/list` 发现、`tools/call` 路由等所有协议细节。完全无需手写 JSON-RPC 桥。

支持两种 server 类型（参见 SDK [session.py](.venv/lib/python3.13/site-packages/copilot/session.py#L733)）：
- **`MCPStdioServerConfig`**：本地子进程 MCP（`command` + `args` + `env`）
- **`MCPHTTPServerConfig`**：远程 HTTP / SSE MCP（`url` + `headers`）

下方示例演示直接对接 Microsoft Learn 官方 MCP HTTP 端点。

In [ ]:
# 💡 SDK 原生 MCP 接入：无需任何手写 JSON-RPC，CLI 自动完成 server 拉起、tools/list 与 tools/call 路由
async def run_native_mcp_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            # ✨ 关键：原生 MCP server 配置
            mcp_servers={
                # 示例 A：远程 HTTP/SSE MCP（如 Microsoft Learn 官方 MCP）
                'microsoft-learn': {
                    'type': 'http',
                    'url': 'https://learn.microsoft.com/api/mcp',
                    'tools': ['*'],   # '*' 暴露全部工具；也可用 ['search_docs'] 白名单
                },
                # 示例 B：本地 stdio MCP（例如官方 filesystem MCP server）
                # 'fs': {
                #     'type': 'stdio',
                #     'command': 'npx',
                #     'args': ['-y', '@modelcontextprotocol/server-filesystem', '/tmp'],
                #     'tools': ['*'],
                # },
            },
        ) as session:
            response = await session.send_and_wait(
                'Use the Microsoft Learn MCP to find the latest Model Context Protocol overview.',
                timeout=120.0,
            )
            if response and hasattr(response.data, 'content'):
                print('[💬 Native MCP - 最终回复]:', response.data.content)

await run_native_mcp_demo()

## 3. 覆写 built-in 工具（`overrides_built_in_tool=True`）

CLI 自带 `read_file` / `edit_file` / `bash` 等内建工具。若想用自己的实现替换它们（例如审计日志、安全沙箱、远程文件系统），**必须**显式声明 `overrides_built_in_tool=True`，否则 SDK 会因命名冲突抛错。

In [ ]:
# 💡 演示：覆写内建的 read_file 工具，附带审计日志
class ReadFileParams(BaseModel):
    path: str = Field(description='Absolute or relative file path to read')

@define_tool(
    name='read_file',
    description='Read a file with custom auditing wrapper',
    overrides_built_in_tool=True,  # ✨ 关键：声明覆写内建同名工具
)
async def custom_read_file(params: ReadFileParams) -> str:
    print(f'[🔒 审计日志] 拦截到 read_file 请求 -> {params.path}')
    # 模拟返回（不实际读盘，保证 demo 安全可复现）
    return f'<MOCK CONTENT of {params.path}: Hello from custom override!>'

async def run_override_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[custom_read_file],
        ) as session:
            response = await session.send_and_wait(
                'Use read_file to read "/etc/hosts" and summarize what you got in one sentence.'
            )
            if response and hasattr(response.data, 'content'):
                print('\n[💬 最终回复]:', response.data.content)

await run_override_demo()

## 4. 跳过权限提示（`skip_permission=True`）

只读 / 安全工具可声明 `skip_permission=True`，直接绕过 `on_permission_request` 提示，提升交互体验。适用于「查询、计算、获取常量」等无副作用的场景。

In [ ]:
# 💡 演示：声明 skip_permission=True，跳过权限请求回调
class SafeLookupParams(BaseModel):
    key: str = Field(description='Config key to look up, e.g. "version"')

@define_tool(
    description='Safe read-only config lookup',
    skip_permission=True,  # ✨ 关键：跳过权限提示
)
async def safe_lookup(params: SafeLookupParams) -> str:
    print(f'[✅ skip_permission 生效] 直接执行，未触发 on_permission_request -> key={params.key}')
    return f'config[{params.key}] = "v2.5.0"'

# 故意拒绝所有权限：如果工具仍然被执行，就说明 skip_permission 真的生效了
async def deny_all(_request):
    print('[🚫 on_permission_request 被调用 —— 不应出现此条]')
    return {'kind': 'deny'}

async def run_skip_permission_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=deny_all,   # 注意这里是 deny_all
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[safe_lookup],
        ) as session:
            response = await session.send_and_wait(
                'Use safe_lookup to retrieve config key "version", then report it.'
            )
            if response and hasattr(response.data, 'content'):
                print('\n[💬 最终回复]:', response.data.content)

await run_skip_permission_demo()

## 5. 工具白名单 / 黑名单（`available_tools` / `excluded_tools`）

在 `create_session()` 中精确控制本次会话可见的**内建**工具集合：

- `available_tools=['read_file', 'lookup_issue']` —— 仅允许列表内（优先级高）
- `excluded_tools=['bash']` —— 屏蔽危险工具（当 `available_tools` 为空时生效）

In [ ]:
# 💡 演示：屏蔽内建 bash / write_file 等危险工具，限定为只读
async def run_excluded_tools_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[lookup_issue],
            # ✨ 黑名单：屏蔽 bash / write_file，模型即使想用也调不到
            excluded_tools=['bash', 'write_file', 'edit_file'],
        ) as session:
            response = await session.send_and_wait(
                'List which tools you have available, then call lookup_issue on AGILE-999 and summarize.'
            )
            if response and hasattr(response.data, 'content'):
                print('\n[💬 黑名单模式 - 最终回复]:', response.data.content)

await run_excluded_tools_demo()

## 6. Declaration-only 工具（外部异步执行）

通过 `Tool(handler=None, ...)` 仅向 LLM **声明**工具 schema，但**不自动执行**。当 LLM 决定调用时，SDK 会广播 `external_tool.requested` 事件，由您的客户端代码异步处理（可跨进程、跨网络、慢工具），最后通过 `session.rpc.tools.handle_pending_tool_call` 把结果回填。

```mermaid
sequenceDiagram
    autonumber
    participant LLM
    participant CLI as copilot CLI
    participant SDK as Python SDK
    participant App as 你的事件订阅器
    participant Worker as 外部执行器<br/>(k8s job / 队列消费者 / 真人)

    LLM->>CLI: tool_call(name=deploy_app, args)
    CLI->>SDK: 检测到 handler=None
    SDK-->>App: event: external_tool.requested<br/>(含 request_id)
    App->>Worker: 投递任务 (HTTP/队列/...)
    Note over Worker: 可耗时数秒~数小时
    Worker-->>App: 任务完成 + 结果
    App->>SDK: session.rpc.tools.handle_pending_tool_call(<br/>request_id, ToolCallResult(...))
    SDK->>CLI: rpc: tools.handlePendingToolCall
    CLI->>LLM: 把 tool 结果回灌
    LLM-->>CLI: 最终回复
```

> 适合长耗时（如 build/deploy）、跨进程（如转发到另一个服务）或需要人在回路决策的工具。
> 对比 `@define_tool`：后者是**同步**调用，SDK 等你的 Python 函数 return；declaration-only 是**异步**，SDK 立即返回并发事件。


In [ ]:
# 💡 演示：声明 handler=None 的工具 + 异步处理 external_tool.requested
from copilot.generated.session_events import ExternalToolRequestedData
from copilot.generated.rpc import ToolsHandlePendingToolCallRequest, ToolCallResult

# 1) Declaration-only：仅声明 schema，无 handler
declaration_only_tool = Tool(
    name='deploy_app',
    description='Deploy an application to a target environment (long-running)',
    parameters={
        'type': 'object',
        'properties': {
            'app': {'type': 'string', 'description': 'App name'},
            'env': {'type': 'string', 'description': 'Target environment'},
        },
        'required': ['app', 'env'],
    },
    handler=None,  # ✨ 关键：无 handler，由外部异步处理
)

async def run_declaration_only_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            tools=[declaration_only_tool],
        ) as session:
            done = asyncio.Event()
            final_text = {'value': None}

            def on_event(event):
                match event.data:
                    case ExternalToolRequestedData() as d:
                        print(f'[📡 external_tool.requested] LLM 请求执行 {d.tool_name}, 参数: {d.arguments}')
                        # 2) 外部异步「真正」执行（这里模拟一个慢任务）
                        async def execute_and_respond():
                            await asyncio.sleep(1.0)  # 模拟慢工具
                            result_text = (
                                f"✅ Deployed app={d.arguments.get('app')} "
                                f"to env={d.arguments.get('env')} in 1.0s"
                            )
                            # 3) 通过 handle_pending_tool_call 把结果回填给 LLM
                            await session.rpc.tools.handle_pending_tool_call(
                                ToolsHandlePendingToolCallRequest(
                                    request_id=d.request_id,
                                    result=ToolCallResult(
                                        text_result_for_llm=result_text,
                                        result_type='success',
                                    ),
                                )
                            )
                            print(f'[✅ handle_pending_tool_call] 已回填结果: {result_text}')
                        asyncio.ensure_future(execute_and_respond())
                    case AssistantMessageData() as d:
                        if d.content:
                            final_text['value'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send('Use deploy_app to deploy app="payments-api" to env="staging", then summarize the result.')
            await asyncio.wait_for(done.wait(), timeout=60.0)
            print('\n[💬 最终回复]:', final_text['value'])

await run_declaration_only_demo()

## 7. Takeaways

- 优先 `@define_tool` + Pydantic，schema 自动且类型安全
- 覆写 built-in 必须显式 `overrides_built_in_tool=True` opt-in；安全工具用 `skip_permission=True` 提升 UX
- `available_tools` / `excluded_tools` 在会话层管控内建工具可见性
- declaration-only（`handler=None`）+ `external_tool.requested` 是处理「慢工具 / 跨进程工具」的官方解法
- MCP 用原生 `mcp_servers={...}`，**无需**手写 JSON-RPC 桥